# P10B — Audit data dan validasi model
Bandingkan pembagian acak dengan validasi kelompok berdasarkan batch. Seluruh imputasi/pemodelan berada dalam pipeline. Hasil menunjukkan sensitivitas terhadap rancangan validasi pada DGP sintetis ini; bukan bukti bahwa satu split tertentu selalu benar.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

data = pd.read_csv('Bab10_Paket/Data/concrete_strength_synthetic.csv')
target = 'strength_MPa'
groups = data['batch_id']
drop_cols = ['specimen_id','batch_id',target]
X = data.drop(columns=drop_cols)
y = data[target]

print('Ukuran:', data.shape, '| batch:', groups.nunique())
print('Duplikasi baris:', data.duplicated().sum())
print('Missing per kolom:')
print(data.isna().sum().to_string())
print('Rentang dan ringkasan:')
print(data.describe().T[['mean','std','min','50%','max']].round(2).to_string())
print('Korelasi Pearson target:')
print(data.select_dtypes('number').corr()['strength_MPa'].sort_values(ascending=False).round(3).to_string())

Ukuran: (990, 9) | batch: 9
Duplikasi baris: 0
Missing per kolom:
specimen_id               0
batch_id                  0
binder_kg_m3              0
water_kg_m3               0
age_day                   0
fine_aggregate_kg_m3      0
coarse_aggregate_kg_m3    0
curing_temp_C             0
strength_MPa              0
Rentang dan ringkasan:
                          mean     std     min     50%      max
binder_kg_m3            394.85   64.26  280.14  397.29   499.98
water_kg_m3             182.36   21.37  145.01  182.50   219.98
age_day                  37.02   25.20    7.00   28.00    90.00
fine_aggregate_kg_m3    752.31   86.29  600.03  752.46   899.93
coarse_aggregate_kg_m3  972.85  101.23  800.69  971.51  1149.93
curing_temp_C            24.74    2.28   17.97   24.68    32.07
strength_MPa             54.28    9.03   28.27   54.48    78.78
Korelasi Pearson target:
strength_MPa              1.000
binder_kg_m3              0.806
age_day                   0.316
fine_aggregate_kg_m3      

In [2]:
splits = {}
idx = np.arange(len(data))
train_i, test_i = train_test_split(idx, test_size=.25, random_state=1010)
splits['Random holdout'] = (train_i, test_i)
gss = GroupShuffleSplit(n_splits=1, test_size=.25, random_state=1010)
gtrain, gtest = next(gss.split(X, y, groups=groups))
splits['Held-out batches'] = (gtrain, gtest)
for name, (tr, te) in splits.items():
    overlap = set(groups.iloc[tr]) & set(groups.iloc[te])
    print(name, '| train=', len(tr), '| test=', len(te), '| batch overlap=', len(overlap),
          '| test batches=', sorted(groups.iloc[te].unique()))

Random holdout | train= 742 | test= 248 | batch overlap= 9 | test batches= ['B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B09']
Held-out batches | train= 660 | test= 330 | batch overlap= 0 | test batches= ['B02', 'B08', 'B09']


In [3]:
models = {
    'Mean baseline': DummyRegressor(strategy='mean'),
    'Ridge': make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), Ridge(alpha=1.0)),
    'Random Forest': make_pipeline(SimpleImputer(strategy='median'),
        RandomForestRegressor(n_estimators=250, min_samples_leaf=3, random_state=1010, n_jobs=-1))
}
records=[]
predictions={}
for split_name, (tr, te) in splits.items():
    for model_name, estimator in models.items():
        estimator.fit(X.iloc[tr], y.iloc[tr])
        pred=estimator.predict(X.iloc[te])
        predictions[(split_name,model_name)] = (te,pred)
        records.append({'split':split_name,'model':model_name,
            'MAE_MPa':mean_absolute_error(y.iloc[te],pred),
            'RMSE_MPa':mean_squared_error(y.iloc[te],pred)**0.5,
            'R2':r2_score(y.iloc[te],pred)})
scores=pd.DataFrame(records)
print(scores.round(3).to_string(index=False))

           split         model  MAE_MPa  RMSE_MPa     R2
  Random holdout Mean baseline    7.656     9.078 -0.001
  Random holdout         Ridge    3.507     4.264  0.779
  Random holdout Random Forest    3.840     4.647  0.738
Held-out batches Mean baseline    7.772     9.367 -0.047
Held-out batches         Ridge    3.956     4.771  0.728
Held-out batches Random Forest    3.958     4.828  0.722


In [4]:
# Diagnostik residual dan biaya kesalahan pada model kompleks
for split_name in splits:
    te,pred=predictions[(split_name,'Random Forest')]
    audit=pd.DataFrame({'batch_id':groups.iloc[te].to_numpy(),
                        'observed':y.iloc[te].to_numpy(),
                        'predicted':pred})
    audit['residual']=audit.observed-audit.predicted
    print('\n',split_name,'| mean residual absolut per batch (MPa)')
    print(audit.groupby('batch_id').residual.apply(lambda z: np.mean(np.abs(z))).round(2).to_string())
    print('95th percentile absolute error:', round(np.quantile(np.abs(audit.residual),.95),2),'MPa')

print('\nKesimpulan harus merujuk populasi penggunaan dan satuan kelompok; split acak tidak menguji transfer ke batch baru.')


 Random holdout | mean residual absolut per batch (MPa)
batch_id
B01    4.76
B02    3.83
B03    2.71
B04    3.58
B05    3.45
B06    5.34
B07    3.53
B08    4.74
B09    2.71
95th percentile absolute error: 9.26 MPa

 Held-out batches | mean residual absolut per batch (MPa)
batch_id
B02    3.97
B08    5.12
B09    2.79
95th percentile absolute error: 8.77 MPa

Kesimpulan harus merujuk populasi penggunaan dan satuan kelompok; split acak tidak menguji transfer ke batch baru.
